In [ ]:
from os.path import exists, sep
from os import listdir, getenv
from re import compile, match, findall
import pandas as pd
from PIL import Image
import numpy as np
from math import floor

EU4_DIR = getenv("EU4_INSTALL_LOCATION")
SEPARATOR = sep
MAP_DIR = EU4_DIR + SEPARATOR + 'map' + SEPARATOR

In [2]:
listdir(MAP_DIR)

['adjacencies.csv',
 'ambient_object.txt',
 'area.txt',
 'climate.txt',
 'continent.txt',
 'default.map',
 'definition.csv',
 'heightmap.bmp',
 'kiel_canal_river.bmp',
 'lakes',
 'minimap_base.bmp',
 'panama_canal_river.bmp',
 'positions.txt',
 'provincegroup.txt',
 'provinces.bmp',
 'random',
 'region.txt',
 'rivers.bmp',
 'seasons.txt',
 'suez_canal_river.bmp',
 'superregion.txt',
 'terrain',
 'terrain.bmp',
 'terrain.txt',
 'trade_winds.txt',
 'trees.bmp',
 'world_normal.bmp']

In [3]:
with open(MAP_DIR + 'definition.csv') as province_colour_values:
    data = []
    column_names = ['Id', 'Name', 'Red', 'Green', 'Blue']
    for line in province_colour_values.readlines():
        if line.startswith('province'):
            continue # First line defines the legend: province;red;green;blue;x;
        parts = line.strip().split(';')
        assert(len(parts) == 6)
        province_id = int(parts[0])
        red_value = int(parts[1])
        green_value = int(parts[2])
        blue_value = int(parts[3])
        province_name = parts[4]
        data.append([province_id, province_name, red_value, green_value, blue_value])
df_province_colours = pd.DataFrame(data=data, columns=column_names)
df_province_colours= df_province_colours.set_index('Id') 
df_province_colours

,Name,Red,Green,Blue
Id,,,,
1,Stockholm,128,34,64
2,Östergötland,0,36,128
3,Småland,128,38,192
4,Bergslagen,0,40,255
5,Värmland,128,42,0
...,...,...,...,...
4937,Lau,210,20,121
4938,Vanua Levu,210,50,161
4939,Te Moana-a-Toi,210,80,11


In [4]:
province_bitmap = Image.open(MAP_DIR + 'provinces.bmp')
array_from_img = np.array(province_bitmap)
height, width, channels = array_from_img.shape
data = np.ndarray((height * width, channels + 2))
x = 0
y = 0
end_of_row = False
end_of_column = False
while y != height:
    colour = array_from_img[y,x,:]
    data[x + y * width, 0] = colour[0]
    data[x + y * width, 1] = colour[1]
    data[x + y * width, 2] = colour[2]
    data[x + y * width, 3] = x
    data[x + y * width, 4] = y
    x += 1
    if (x == width):
        x = 0
        y += 1
print(data)

[[4.500e+01 2.030e+02 2.030e+02 0.000e+00 0.000e+00]
 [4.500e+01 2.030e+02 2.030e+02 1.000e+00 0.000e+00]
 [4.500e+01 2.030e+02 2.030e+02 2.000e+00 0.000e+00]
 ...
 [5.100e+01 2.210e+02 2.510e+02 5.629e+03 2.047e+03]
 [5.100e+01 2.210e+02 2.510e+02 5.630e+03 2.047e+03]
 [5.100e+01 2.210e+02 2.510e+02 5.631e+03 2.047e+03]]


In [5]:
df_colour_positions = pd.DataFrame(data, columns=['Red', 'Green', 'Blue', 'X', 'Y'])
df_colour_positions = df_colour_positions.astype(int)
df_colour_positions.head(10)

,Red,Green,Blue,X,Y
0,45,203,203,0,0
1,45,203,203,1,0
2,45,203,203,2,0
3,45,203,203,3,0
4,45,203,203,4,0
5,45,203,203,5,0
6,45,203,203,6,0
7,45,203,203,7,0
8,45,203,203,8,0
9,45,203,203,9,0


In [6]:
df_provinces = pd.read_csv('./data/provinces_stage1.csv', index_col=0) # Imported from file that collects province data from history file
for idx, row in df_province_colours.iterrows():
    matched_red = df_colour_positions.loc[df_colour_positions['Red'] == row.Red,:]
    matched_green = matched_red.loc[matched_red['Green'] == row.Green,:]
    matched_pixels = matched_green.loc[matched_green['Blue'] == row.Blue,:]
    matched_pixels['Province_Id'] = idx
    try:
        left_edge = min(matched_pixels['X'].values)
        right_edge = max(matched_pixels['X'].values)
        top_edge = max(matched_pixels['Y'].values)
        bottom_edge = min(matched_pixels['Y'].values)
        middle = (floor((right_edge + left_edge) / 2), floor((top_edge + bottom_edge) / 2))
        middle_X, middle_Y = middle
        df_provinces.loc[df_provinces['id'] == idx, 'middle_x_pos'] = middle_X
        df_provinces.loc[df_provinces['id'] == idx, 'middle_y_pos'] = middle_Y
    except:
        assert(matched_pixels.shape[0] < 1) # Verify the matched pixels are actually empty
    # print(f"Top: {top_edge}\nBottom: {bottom_edge}\nLeft: {left_edge}\nRight: {right_edge}\n--------\nMiddle point: {middle}\n")

In [7]:
df_provinces.info()
df_provinces["hre"] = df_provinces["hre"] == "yes"
df_provinces["is_city"] = df_provinces["is_city"] == "yes"
df_provinces["hre"] = df_provinces["hre"].astype(bool)
df_provinces["is_city"] = df_provinces["is_city"].astype(bool)

<class 'pandas.DataFrame'>
RangeIndex: 3923 entries, 0 to 3922
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   owner            3923 non-null   str    
 1   culture          3923 non-null   str    
 2   religion         3923 non-null   str    
 3   hre              3923 non-null   str    
 4   base_tax         3923 non-null   int64  
 5   base_production  3923 non-null   int64  
 6   trade_goods      3162 non-null   str    
 7   base_manpower    3923 non-null   int64  
 8   capital          3923 non-null   str    
 9   is_city          3923 non-null   str    
 10  center_of_trade  3923 non-null   int64  
 11  name             3923 non-null   str    
 12  id               3923 non-null   int64  
 13  trade_node       3174 non-null   str    
 14  middle_x_pos     3923 non-null   float64
 15  middle_y_pos     3923 non-null   float64
dtypes: float64(2), int64(5), str(9)
memory usage: 490.5 KB


In [8]:
df_provinces = df_provinces.rename(columns={"id": "province_id"})
df_provinces.to_csv("./data/provinces_stage2.csv")
df_provinces

,owner,culture,religion,hre,base_tax,base_production,trade_goods,base_manpower,capital,is_city,center_of_trade,name,province_id,trade_node,middle_x_pos,middle_y_pos
0,SWE,swedish,catholic,False,5,5,grain,3,Stockholm,True,2,Uppland,1,baltic_sea,3077.0,321.0
1,NOR,Unowned,catholic,False,1,1,fur,1,Frösön,True,0,Jamtland,10,north_sea,3029.0,234.0
2,FRI,frisian,catholic,True,6,6,livestock,2,Leeuwarden,True,0,Friesland,100,english_channel,2896.0,465.0
3,ENG,cree,totemism,False,1,2,unknown,1,Cree,True,0,Cree,1000,james_bay,1579.0,228.0
4,FRA,anishinabe,totemism,False,2,2,unknown,1,Sault,True,0,Sault,1001,ohio,1510.0,368.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3918,FRA,innu,totemism,False,2,2,unknown,1,Tadoussac,True,0,Atiamekw,995,st_lawrence,1721.0,335.0
3919,Unowned,innu,totemism,False,1,1,unknown,1,Innu,False,0,Innu,996,st_lawrence,1892.0,283.0
3920,Unowned,innu,totemism,False,1,1,unknown,1,Labrador,False,0,Labrador,997,st_lawrence,1914.0,246.0
3921,GBR,inuit,totemism,False,1,2,unknown,1,Ungava,True,0,Ungava,998,james_bay,1619.0,165.0
